# LEVELS Benchmark — Level 1: Atomic

Single DSI/API call from natural language. Each task is scored by 7 binary checks: tool, args, path, count, keywords, forbidden keywords and nonempty response. `score` is the fraction of checks passed, and `passed` means all 7 passed. A judge ensemble also scores efficiency against the human `reference_path`.

| Level | Notebook |
|---|---|
| 1 Atomic | `LEVELS_1_atomic.ipynb` |
| 2 Workflow | `LEVELS_2_workflow.ipynb` |
| 3 Discovery | `LEVELS_3_discovery.ipynb` |
| Calibration (RQ4) | `LEVELS_4_calibration.ipynb` |
| Prompt phrasing | `LEVELS_5_prompt_variants.ipynb` |
| Cross-level RQ summary | `LEVELS_6_summary.ipynb` |

### Data Management Categories

1. Search: external/domain knowledge and literature
2. Data Discovery: locating datasets, files, or records in DSI repositories
3. Data Preparation: cleaning, validating, standardizing scientific data
4. Metadata Generation: structured documentation and provenance
5. Analysis & Summarization: interpreting, aggregating, synthesizing data

### MCP mode

To benchmark through MCP (atomic tools served by `levels_mcp_server.py`): run `python levels_mcp_server.py` in a terminal, then pass `use_mcp=True` to `init(...)`. Inspector: `npx @modelcontextprotocol/inspector` → Streamable HTTP → `http://127.0.0.1:8000/mcp`.

Shared harness code (scoring, parallel runner, reports) lives in `levels_bench/`, and the task
definitions in `levels_bench/tasks/`. Each (model, task) pair runs as an independent job in a
pool of worker processes, so `max_workers` runs proceed at once. Every finished result is
appended to `levels_benchmark_results/<run_id>_<label>.partial.jsonl` as it completes. If a run
is interrupted, call `run_benchmark(..., resume=True)` to run only what's missing.

## Setup

In [ ]:
# Run once per environment (not needed on every run):
# %pip install -q -r requirements.txt
# %pip install -q --no-build-isolation -e ../..

In [ ]:
from levels_bench import init, run_benchmark, flatten, results_to_dataframe, load_results, reports, debug_task
from levels_bench.tasks import LEVEL1_TASKS, get_task

cfg = init(
    # models_to_run=["aws-gov.gpt-5.4"],   # default: every model in levels_bench.config.POSSIBLE_MODELS
    # judge_models=[...],                   # default: same list; judged scores are averaged across them
    max_workers=4,                          # concurrent agent runs; raise if the portal's rate limit allows
    task_timeout_sec=180,                   # per agent turn
    # skip_network=True,                    # skip tasks that download from oceans11.lanl.gov
)

In [ ]:
reports.task_table(LEVEL1_TASKS)

## Run

In [ ]:
l1_results = flatten(run_benchmark(LEVEL1_TASKS, "level1_atomic"))
# To analyze an earlier run without re-running it:
# l1_results = load_results("level1_atomic")
l1_df = results_to_dataframe(l1_results)
l1_df

## Results

In [ ]:
reports.model_summary(l1_df)

In [ ]:
reports.failures(l1_df)

In [ ]:
reports.timing_breakdown(l1_results)

## Inspect / debug

Transcripts are recorded for every run. `debug_task` reruns one task in this process with live output.

In [ ]:
# reports.transcript(l1_results, "L1_load_ocean11")
# tr = debug_task(get_task("L1_load_ocean11"), model="anthropic.claude-sonnet-4-5-20250929-v1:0")